# Module 08 — Graph Algorithms Traversals and DAGs

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_num_islands import num_islands
from p02_count_components import count_components
from p03_topological_order import topological_order

print("module 08: Graph Algorithms Traversals and DAGs")
print("problems available:", 8)
for name in ['p01_num_islands', 'p02_count_components', 'p03_topological_order', 'p04_has_cycle_directed', 'p05_can_finish_courses', 'p06_rotting_oranges', 'p07_shortest_path_grid', 'p08_word_ladder']:
    print(f"  {name}")

## 1. Baseline — `p01_num_islands`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert num_islands([["1", "1", "0"], ["1", "0", "0"], ["0", "0", "1"]]) == 2
assert num_islands([]) == 0
assert num_islands([[]]) == 0
assert num_islands([["0"]]) == 0
assert num_islands([["1"]]) == 1
# All land is one island.
assert num_islands([["1", "1"], ["1", "1"]]) == 1
# Diagonal cells are NOT connected.
assert num_islands([["1", "0"], ["0", "1"]]) == 2
# A checkerboard is all singletons.
grid = [["1" if (r + c) % 2 == 0 else "0" for c in range(4)] for r in range(4)]
assert num_islands(grid) == 8
# The input must not be mutated.
original = [["1", "1"], ["0", "1"]]
snapshot = [row[:] for row in original]
num_islands(original)
assert original == snapshot, "num_islands must not destroy its argument"
# Scale: 300x300 of solid land must not recurse to death.
big = [["1"] * 300 for _ in range(300)]
assert num_islands(big) == 1

print("all assertions held")

## 2. Predict before you run

A directed graph has edges 0→1, 0→2, 1→3, 2→3. Predict whether it contains a cycle. Then predict what a DFS that flags any already-visited node would report.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert count_components(5, [(0, 1), (1, 2), (3, 4)]) == 2
assert count_components(5, [(0, 1), (1, 2), (2, 3), (3, 4)]) == 1
assert count_components(0, []) == 0
# No edges at all: every node is its own component.
assert count_components(4, []) == 4
assert count_components(1, []) == 1
# A self-loop does not connect anything new.
assert count_components(3, [(0, 0)]) == 3
# Duplicate edges must not change the count.
assert count_components(3, [(0, 1), (0, 1), (1, 0)]) == 2
# A cycle is still one component.
assert count_components(3, [(0, 1), (1, 2), (2, 0)]) == 1
# Scale.
chain = [(i, i + 1) for i in range(99_999)]
assert count_components(100_000, chain) == 1

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert topological_order(4, [(0, 1), (1, 2), (2, 3)]) == [0, 1, 2, 3]
# A cycle yields the empty list.
assert topological_order(2, [(0, 1), (1, 0)]) == []
assert topological_order(3, [(0, 1), (1, 2), (2, 0)]) == []
# A self-loop is a cycle.
assert topological_order(1, [(0, 0)]) == []
# No edges: smallest-first gives sorted order.
assert topological_order(3, []) == [0, 1, 2]
assert topological_order(0, []) == []
# A diamond is a valid DAG, not a cycle - this is the case a, # two-state visited check wrongly rejects.
assert topological_order(4, [(0, 1), (0, 2), (1, 3), (2, 3)]) == [0, 1, 2, 3]
# Any returned order must respect every edge.
edges = [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]
order = topological_order(6, edges)
assert len(order) == 6
pos = {node: i for i, node in enumerate(order)}
assert all(pos[u] < pos[v] for u, v in edges)

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. Unweighted shortest path means BFS. DFS's first arrival is not optimal.
2. Directed cycle detection needs three states; two produce false positives on any diamond.
3. Kahn's algorithm detects cycles for free - check the output length.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem